In [1]:
import pandas as pd
import duckdb

df = pd.read_csv(
    "/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv"
)

con = duckdb.connect()
con.register("diabetes_data", df)

print("SQL setup ready!")

SQL setup ready!


In [2]:
result = con.execute("""
SELECT 
    COUNT(*) AS readmitted_under_30_days
FROM diabetes_data
WHERE readmitted = '<30';
""").fetchone()

print("Readmitted within 30 days:", result[0])

Readmitted within 30 days: 11357


In [3]:
result = con.execute("""
SELECT 
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) 
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data;
""").fetchone()

print("30-day readmission rate:", result[0], "%")

30-day readmission rate: 11.16 %


In [4]:
result = con.execute("""
SELECT
    readmitted,
    COUNT(*) AS encounters,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM diabetes_data
GROUP BY readmitted
ORDER BY encounters DESC;
""").df()

result

,readmitted,encounters,percentage
0,NO,54864,53.91
1,>30,35545,34.93
2,<30,11357,11.16


In [5]:
result = con.execute("""
SELECT
    age,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) 
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY age
ORDER BY readmission_rate DESC;
""").df()

result

,age,total_encounters,readmitted_under_30,readmission_rate
0,[20-30),1657,236.0,14.24
1,[80-90),17197,2078.0,12.08
2,[70-80),26068,3069.0,11.77
3,[30-40),3775,424.0,11.23
4,[60-70),22483,2502.0,11.13
5,[90-100),2793,310.0,11.10
6,[40-50),9685,1027.0,10.60
7,[50-60),17256,1668.0,9.67
8,[10-20),691,40.0,5.79
9,[0-10),161,3.0,1.86


In [6]:
result = con.execute("""
SELECT
    number_inpatient,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY number_inpatient
ORDER BY number_inpatient;
""").df()

result

,number_inpatient,total_encounters,readmitted_under_30,readmission_rate
0,0,67630,5706.0,8.44
1,1,19521,2523.0,12.92
2,2,7566,1319.0,17.43
3,3,3411,692.0,20.29
4,4,1622,383.0,23.61
5,5,812,255.0,31.40
6,6,480,166.0,34.58
7,7,268,95.0,35.45
8,8,151,67.0,44.37
9,9,111,47.0,42.34


In [7]:
result = con.execute("""
SELECT
    time_in_hospital,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY time_in_hospital
ORDER BY time_in_hospital;
""").df()

result

,time_in_hospital,total_encounters,readmitted_under_30,readmission_rate
0,1,14208,1162.0,8.18
1,2,17224,1712.0,9.94
2,3,17756,1894.0,10.67
3,4,13924,1644.0,11.81
4,5,9966,1199.0,12.03
5,6,7539,949.0,12.59
6,7,5859,752.0,12.83
7,8,4391,625.0,14.23
8,9,3002,412.0,13.72
9,10,2342,336.0,14.35


In [8]:
SELECT
    diag_1,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
WHERE diag_1 != '?'
GROUP BY diag_1
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;

IndentationError: unexpected indent (349157836.py, line 2)

In [ ]:
result = con.execute("""
SELECT
    diag_1,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
WHERE diag_1 != '?'
GROUP BY diag_1
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    CASE
        WHEN diag_1 LIKE 'V%' THEN 'Supplementary V-Code'
        WHEN diag_1 LIKE 'E%' THEN 'External Cause'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 1 AND 139
            THEN 'Infectious & Parasitic'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 140 AND 239
            THEN 'Neoplasms'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 240 AND 279
            THEN 'Endocrine & Metabolic'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 280 AND 289
            THEN 'Blood Disorders'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 290 AND 319
            THEN 'Mental Disorders'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 320 AND 389
            THEN 'Nervous System'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 390 AND 459
            THEN 'Circulatory'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 460 AND 519
            THEN 'Respiratory'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 520 AND 579
            THEN 'Digestive'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 580 AND 629
            THEN 'Genitourinary'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 630 AND 679
            THEN 'Pregnancy'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 680 AND 709
            THEN 'Skin & Subcutaneous'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 710 AND 739
            THEN 'Musculoskeletal'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 740 AND 759
            THEN 'Congenital'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 760 AND 779
            THEN 'Perinatal'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 780 AND 799
            THEN 'Symptoms & Signs'
        WHEN TRY_CAST(split_part(diag_1, '.', 1) AS INTEGER) BETWEEN 800 AND 999
            THEN 'Injury & Poisoning'
        ELSE 'Other'
    END AS diagnosis_category,

    COUNT(*) AS total_encounters,

    SUM(
        CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
    ) AS readmitted_under_30,

    ROUND(
        100.0 * SUM(
            CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS readmission_rate

FROM diabetes_data

WHERE diag_1 != '?'

GROUP BY diagnosis_category

HAVING COUNT(*) >= 100

ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    payer_code,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
WHERE payer_code != '?'
GROUP BY payer_code
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    discharge_disposition_id,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY discharge_disposition_id
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    CASE
        WHEN number_inpatient = 0 THEN 'No prior inpatient visits'
        WHEN number_inpatient = 1 THEN '1 prior inpatient visit'
        WHEN number_inpatient BETWEEN 2 AND 3 THEN '2-3 prior inpatient visits'
        ELSE '4+ prior inpatient visits'
    END AS prior_visit_group,

    COUNT(*) AS total_encounters,

    SUM(
        CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
    ) AS readmitted_under_30,

    ROUND(
        100.0 * SUM(
            CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS readmission_rate

FROM diabetes_data

GROUP BY prior_visit_group

ORDER BY
    CASE prior_visit_group
        WHEN 'No prior inpatient visits' THEN 1
        WHEN '1 prior inpatient visit' THEN 2
        WHEN '2-3 prior inpatient visits' THEN 3
        ELSE 4
    END;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    gender,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
WHERE gender != 'Unknown/Invalid'
GROUP BY gender
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    admission_type_id,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY admission_type_id
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    change,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY change
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    A1Cresult,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY A1Cresult
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    number_emergency,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY number_emergency
ORDER BY number_emergency;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    number_outpatient,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY number_outpatient
ORDER BY number_outpatient;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    number_diagnoses,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
GROUP BY number_diagnoses
ORDER BY number_diagnoses;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    medical_specialty,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END) AS readmitted_under_30,
    ROUND(
        100.0 * SUM(CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS readmission_rate
FROM diabetes_data
WHERE medical_specialty != '?'
GROUP BY medical_specialty
HAVING COUNT(*) >= 100
ORDER BY readmission_rate DESC;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    CASE
        WHEN number_inpatient = 0
             AND number_emergency = 0
             AND number_outpatient = 0
            THEN 'Low Utilization'

        WHEN number_inpatient <= 1
             AND number_emergency <= 1
             AND number_outpatient <= 2
            THEN 'Moderate Utilization'

        WHEN number_inpatient BETWEEN 2 AND 3
             OR number_emergency BETWEEN 2 AND 3
             OR number_outpatient BETWEEN 3 AND 5
            THEN 'High Utilization'

        ELSE 'Very High Utilization'
    END AS utilization_segment,

    COUNT(*) AS total_encounters,

    SUM(
        CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
    ) AS readmitted_under_30,

    ROUND(
        100.0 * SUM(
            CASE WHEN readmitted = '<30' THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS readmission_rate

FROM diabetes_data

GROUP BY utilization_segment

ORDER BY
    CASE utilization_segment
        WHEN 'Low Utilization' THEN 1
        WHEN 'Moderate Utilization' THEN 2
        WHEN 'High Utilization' THEN 3
        WHEN 'Very High Utilization' THEN 4
    END;
""").df()

result

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW healthcare_analysis AS

SELECT
    encounter_id,
    patient_nbr,
    age,
    gender,
    time_in_hospital,
    admission_type_id,
    discharge_disposition_id,
    payer_code,
    medical_specialty,

    number_inpatient,
    number_emergency,
    number_outpatient,
    number_diagnoses,

    A1Cresult,
    change,

    diag_1,
    diag_2,
    diag_3,

    readmitted,

    CASE
        WHEN readmitted = '<30' THEN 1
        ELSE 0
    END AS readmitted_30_flag,

    CASE
        WHEN number_inpatient = 0
             AND number_emergency = 0
             AND number_outpatient = 0
            THEN 'Low Utilization'

        WHEN number_inpatient <= 1
             AND number_emergency <= 1
             AND number_outpatient <= 2
            THEN 'Moderate Utilization'

        WHEN number_inpatient BETWEEN 2 AND 3
             OR number_emergency BETWEEN 2 AND 3
             OR number_outpatient BETWEEN 3 AND 5
            THEN 'High Utilization'

        ELSE 'Very High Utilization'
    END AS utilization_segment

FROM diabetes_data;
""")

print("Final SQL view created!")

In [ ]:
result = con.execute("""
SELECT *
FROM healthcare_analysis
LIMIT 10;
""").df()

result

In [ ]:
result = con.execute("""
SELECT
    utilization_segment,
    COUNT(*) AS total_encounters,
    SUM(readmitted_30_flag) AS readmitted_30,
    ROUND(
        100.0 * SUM(readmitted_30_flag) / COUNT(*),
        2
    ) AS readmission_rate
FROM healthcare_analysis
GROUP BY utilization_segment
ORDER BY
    CASE utilization_segment
        WHEN 'Low Utilization' THEN 1
        WHEN 'Moderate Utilization' THEN 2
        WHEN 'High Utilization' THEN 3
        WHEN 'Very High Utilization' THEN 4
    END;
""").df()

result

In [ ]:
final_df = con.execute("""
SELECT *
FROM healthcare_analysis
""").df()

final_df.to_csv(
    "/kaggle/working/healthcare_readmission_analysis.csv",
    index=False
)

print("Export complete!")
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))